# Admin: Manual Transaction Logging
Log offline costs (Cicero credits, GCP bills) and inflows (GitHub Sponsors) to the transactions ledger.

In [ ]:
import httpx
import json
from dotenv import load_dotenv

load_dotenv("../.env")

BASE_URL = "http://localhost:8000"
print(f"Target: {BASE_URL}")

## Log an Expense
Edit the fields below and run the cell to insert a transaction.

Common examples:
- Cicero top-up: `source="cicero"`, `billing_model="bulk"`, `type="outflow"`
- GCP monthly bill: `source="gcp"`, `billing_model="subscription"`, `type="outflow"`
- GitHub Sponsors: `source="github_sponsors"`, `billing_model="subscription"`, `type="inflow"`

In [ ]:
# resp = httpx.post(f"{BASE_URL}/api/transactions/", json={
#     "type": "outflow",
#     "source": "example",
#     "billing_model": "bulk",
#     "amount_usd": 00.00,
#     "description": "fake",
# })

# print(f"Status: {resp.status_code}")
# print(json.dumps(resp.json(), indent=2, default=str))

## View Recent Transactions

In [ ]:
resp = httpx.get(f"{BASE_URL}/api/transactions/", params={"limit": 20})
transactions = resp.json()

print(f"Showing {len(transactions)} transactions\n")
print(f"{'ID':>4}  {'Type':<8} {'Source':<18} {'Model':<14} {'Amount':>10}  Description")
print("-" * 90)
for t in transactions:
    print(f"{t['id']:>4}  {t['type']:<8} {t['source']:<18} {t['billing_model']:<14} ${float(t['amount_usd']):>9.4f}  {t.get('description') or ''}")

## Quick Summary by Source

In [ ]:
from collections import defaultdict

totals = defaultdict(float)
for t in transactions:
    sign = 1 if t["type"] == "inflow" else -1
    totals[t["source"]] += sign * float(t["amount_usd"])

print(f"{'Source':<20} {'Net ($)':>10}")
print("-" * 32)
for source, net in sorted(totals.items()):
    print(f"{source:<20} {net:>10.4f}")
print("-" * 32)
print(f"{'TOTAL':<20} {sum(totals.values()):>10.4f}")